# Antarctic Segmentation

Formal experiments 1-14 are complete. This is a lightweight launcher; all execution switches default to False. Results remain in external files.

[Project README](README.md) | [Closeout](reports/CLOSEOUT.md) | [Code map](CODE_MAP.md) | [Experiment index](reports/index.csv)


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").is_dir():
    raise RuntimeError("Start this notebook from the project root")
sys.path.insert(0, str(ROOT / "scripts"))


## Reports

[Experiment 13 report and training curves](reports/exp13/README.md) | [Experiment 14 scale analysis](reports/exp14/README.md) | [Historical overview](reports/overview.jpg)

These are plain links; no large figures or result arrays are rendered or embedded in this notebook.


In [ ]:
import pandas as pd
pd.read_csv(ROOT / "reports/index.csv")

## Cached reports and optional historical evaluation

Report rebuilding uses saved metadata and masks only. It preserves this notebook. Historical rescoring also uses saved masks; a full prediction replay requires a separate explicit switch and may be expensive. None of these operations trains a model.


In [ ]:
RUN_REBUILD_REPORTS = False
RUN_RESCORE_SAVED_MASKS = False
RUN_REPLAY_HISTORICAL_INFERENCE = False

if RUN_RESCORE_SAVED_MASKS:
    from reference_benchmark import rescore_only
    rescored_results = rescore_only()
if RUN_REPLAY_HISTORICAL_INFERENCE:
    from reference_benchmark import run
    historical_results = run()
if RUN_REBUILD_REPORTS or RUN_RESCORE_SAVED_MASKS or RUN_REPLAY_HISTORICAL_INFERENCE:
    from report_experiments import main as build_reports
    build_reports()


## Explicit reproduction only

The retained results are already complete. Enable training/inference below only to reproduce a protocol deliberately. Use a new output directory for changed settings; keep the original directory only for exact-protocol resume. The declined 5x5 preview is archived and has no formal notebook entry.


## Experiment 13: new four-class baseline

Run setup first; restart the kernel after migration. Labels are background=0, penguin=1, other_bird=2, seal=3 and ignore=255. A separate view maps whale 4 to background and recomputes image sampling groups; historical five-class data remains intact.

ResNet34 U-Net: kernel 3, cached ImageNet encoder, random decoder/head, no historical initialization. Ten joint epochs, 640 letterbox, batch 6, AMP, normal BN, AdamW 1e-4/0.01, CE(ignore=255), seed 42. Sampling uses 1/sqrt(group size), replacement, and len(train) draws.

Both switches default to False. Training performs the full audit and discarded real preflight steps first; only OOM allows batch 4. Keep the same run directory to resume. Evaluation uses best.pt, benchmark_v2, native/resize only, 16 far points and six landscape negatives; NJU is qualitative only.

[Full protocol and results](reports/exp13/README.md). Outputs stay under artifacts/. Four-class mIoU is not directly comparable to historical five-class mIoU; this is not a whale-only ablation.


In [ ]:
RUN_TRAINING_13 = False  # Explicit reproduction only
RUN_EVALUATION_13 = False  # Explicit best.pt inference only
EXP13_RUN_DIR = ROOT / "artifacts/experiments/exp13_results/run_20261111_150108"

if RUN_TRAINING_13 or RUN_EVALUATION_13:
    from experiment13 import run_experiment13
    experiment13_results = run_experiment13(
        root=ROOT,
        run_dir=EXP13_RUN_DIR,
        train=RUN_TRAINING_13,
        evaluate=RUN_EVALUATION_13,
    )
else:
    print("Experiment 13 is complete. Read reports/exp13/README.md.")


## Experiment 14: Inference Scale Sensitivity

Fixed experiment-13 best weights; no training. Compare actual 640/960/1280/1920 centered letterbox inputs and Native, all float32 with AMP and TF32 disabled. Only CUDA OOM allows unchanged-size CPU float32 fallback. Metrics use frozen benchmark_v2 and exact existing far points; NJU has visualizations only.

Run the setup cell first. The output directory below supports verified-cache resume. The execution switch stays off to avoid accidental reruns. Results and large figures remain external: [report](reports/exp14/README.md). The complete local gallery also includes the six private NJU photos and must stay under ignored artifacts storage.


In [ ]:
RUN_EXPERIMENT14 = False  # Explicit execution/resume only; no training
EXPERIMENT14_RUN_DIR = ROOT / "artifacts/experiments/exp14_results/run_20261111_175034"

if RUN_EXPERIMENT14:
    from experiment14 import run_experiment14
    experiment14_result = run_experiment14(
        root=ROOT,
        run_dir=EXPERIMENT14_RUN_DIR,
        execute=True,
    )
    print("Report:", experiment14_result["report"])
    print("Local full gallery:", experiment14_result["gallery"])
else:
    print("Experiment 14 is complete. Read reports/exp14/README.md.")
